<a href="https://colab.research.google.com/github/talhanoor23/generative-ai/blob/main/RAGS/MultiQuery_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [ ]:
docs = "lorem ipsum"

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300,
    chunk_overlap = 50
)
splits = splitter.split_documents(docs)

In [ ]:
from langchain.embeddings import OpenAIEmbeddings
embeddings = OpenAIEmbeddings()
######
from langchain.vectorstores import Chroma
db = Chroma.from_documents(splits, embeddings)
retreiver = db.as_retriever()

In [ ]:
from langchain.prompts import ChatPromptTemplate

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five
different versions of the given user question to retrieve relevant documents from a vector
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search.
Provide these alternative questions separated by newlines. Original question: {question}"""
prompt_perspective = ChatPromptTemplate.from_template(template)


In [ ]:
# from langchain_core.output_parsers import StrOutputParser
# from langchain.chat_models import ChatOpenAI
# from langchain.schema.runnable import RunnablePassthrough

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

generate_queries = (
                    prompt_perspective
                    | ChatOpenAI(temprature = 0)
                    | StrOutputParser()
                    | (lambda x: x.split("\n"))
)

In [ ]:
from langchain.load import dumps, loads

def get_unique_union(documents: list[list]):
    """ Unique union of retrieved docs """
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

In [ ]:
retreiver_chain = generate_queries | retreiver.map() | get_unique_union

In [ ]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough

In [ ]:
# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI(temperature=0)

In [ ]:
final_rag_chain = (
    {
        "content" : retreiver_chain,
        "question" : itemgetter("question")
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
question = "nfdg"
final_rag_chain.invoke({"question":question})